# Import Libraries

In [2]:
import numpy as np
import evaluate

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Load Dataset

In [3]:
dataset = load_dataset("SetFit/ag_news")

dataset

train.jsonl:   0%|          | 0.00/33.8M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 7600
    })
})

# Load the Tokenizer

In [4]:
model_name = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

# Tokenize the Dataset

In [5]:
def tokenize_function(example):
  return tokenizer(
      example["text"],
      padding="max_length",
      truncate=True,
      max_length=128
  )

tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

# Preprocess the Dataset

In [6]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

# Reduce the Datset for initial testing

In [11]:
train_dataset = tokenized_dataset["train"]
test_dataset = tokenized_dataset["test"]

# Sequence Classification in BERT

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Define the Evaluation Metrics i.e. Accuracy and F1-Score

In [13]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)

  accuracy = accuracy_metric.compute(
      predictions=predictions,
      references=labels
  )

  f1 = f1_metric.compute(
      predictions=predictions,
      references=labels,
      average="weighted"
  )

  return {
      "accuracy": accuracy["accuracy"],
      "f1": f1["f1"]
  }

# Define the Training Arguments

In [14]:
training_args = TrainingArguments(
    output_dir="./bert-ag-news",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=1000,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# Create the Trainer

In [15]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.187575,0.174183,0.945789,0.945806


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.187575,0.174183,0.945789,0.945806
2,0.131497,0.188689,0.948289,0.948317


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=15000, training_loss=0.17677413075764975, metrics={'train_runtime': 1821.2569, 'train_samples_per_second': 131.777, 'train_steps_per_second': 8.236, 'total_flos': 1.6448049870355584e+16, 'train_loss': 0.17677413075764975, 'epoch': 2.0})

In [18]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.131497,0.188689,2,0.948289,0.948317


{'eval_loss': 0.18868906795978546, 'eval_accuracy': 0.9482894736842106, 'eval_f1': 0.9483169584473546}


# Testing the Model on Custom Headlines

In [19]:
import torch

def predict_news_topic(text):
  label_names = ["World", "Sports", "Business", "Sci/Tech"]
  inputs = tokenizer(
      text,
      truncation=True,
      padding=True,
      return_tensors="pt",
      max_length=128
  )

  inputs = {
      key: value.to(model.device) for key, value in inputs.items()
  }

  outputs = model(**inputs)
  prediction = torch.argmax(outputs.logits, dim=1).item()

  return label_names[prediction]

In [20]:
headline = "UK Scientists Allowed to Clone Human Embryos (Reuters) Reuters - British scientists said on Wednesday\they had received permission to clone human embryos for medical\research, in what they believe to be the first such license to\be granted in Europe."
print(predict_news_topic(headline))

Sci/Tech


# Save the fine-tuned Model

In [ ]:
trainer.save_model("./bert-ag-news-final")
tokenizer.save_pretrained("./bert-ag-news-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('drive/MyDrive/bert-ag-news-final/tokenizer_config.json',
 'drive/MyDrive/bert-ag-news-final/tokenizer.json')